# Provider Adapter
# 0. 介绍
**研究背景**：大模型厂商返回的结果不只有文字，还可能包含`调用哪个工具`、`为什么停止`、`用了多少 Token`等信息。不同模型厂商返回这些信息的格式并不相同，因此 Agent 需要一个统一的适配器，把不同格式整理成相同结构。

**现存问题**：如果程序只读取大模型返回的文字内容，就可能漏掉工具调用。这样，即使大模型已经正确判断出要执行什么操作，外部程序也不知道该做什么，最终导致任务失败，而且很难发现问题出在哪里。

**解决方案**：本 Notebook 将实现一个极简的 Provider Adapter，把模型的原始响应整理成统一的`数据包`。然后用同一份真实 API 响应进行对比：错误版本只读取文字，改进版本读取完整响应，从而直观看到适配器如何避免信息丢失并让任务成功执行。
## 目录
0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证API响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [16]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型要完成的任务，以及程序接收和处理结果的方式。

# 2. 前置准备
## 2.1 说明可用工具
大模型需要先知道自己可以做什么，才能返回可以执行的操作。本节用一份简单的说明告诉大模型，它可以使用 `assign_ticket` 把工单分给退款组或技术组。

In [17]:
tools = [{
    "type": "function",
    "function": {
        "name": "assign_ticket",
        "description": "把工单分给退款组或技术组",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "to": {"type": "string", "enum": ["refund", "tech"]},
            },
            "required": ["ticket_id", "to"],
        }}}]
print(f"可用工具：{tools[0]['function']['name']}")

可用工具：assign_ticket


输出显示 `assign_ticket` 已经准备好，说明大模型知道了工具名称、需要填写的内容和可选的组别。下一步会给大模型一张具体工单，让它决定如何分配。

## 2.2 写出具体任务
有了工具说明，还需要告诉大模型这次要处理什么。本节提供一张重复扣款工单，并要求大模型使用刚才的工具完成分配。

In [18]:
messages = [
    {"role": "system", "content": "你是工单分流助手，只能调用 assign_ticket。"},
    {"role": "user", "content": "工单 A-1042：用户被重复扣款并要求退款，请分配处理组。"},
]
print(f"任务：{messages[-1]['content']}")

任务：工单 A-1042：用户被重复扣款并要求退款，请分配处理组。


输出显示了大模型将要处理的工单，其中包含工单编号、问题和用户要求。下一步会定义一种固定的保存方式，用来接收大模型返回的各项结果。

## 2.3 统一保存结果
大模型返回的结果包含多种信息，如果随意存放，后面的程序就很难统一读取。本节定义一个简单的数据包，把文字、工具调用、停止原因和 Token 用量放在固定位置。

In [19]:
class Packet:
    def __init__(self, content, calls, stop_reason, usage):
        self.content = content          # 文字内容
        self.calls = calls              # 工具调用
        self.stop_reason = stop_reason  # 停止原因
        self.usage = usage              # Token 用量

print("数据包：content, calls, stop_reason, usage")

数据包：content, calls, stop_reason, usage


输出列出了数据包中的四项内容，说明后面无论大模型返回什么格式，程序都会从这些固定位置读取结果。下一步会规定程序收到工具调用后如何改变工单状态。

## 2.4 定义执行方式
工具调用只是大模型提出的操作，程序还需要真正执行它。本节规定收到 `assign_ticket` 后，把工单编号和处理组写入已分配列表。

In [20]:
def execute(call, state):
    if call["name"] == "assign_ticket":
        state["assigned"].append(call["arguments"])  # 记录分配结果

print("执行方式：assign_ticket 写入 assigned")

执行方式：assign_ticket 写入 assigned


输出说明执行方式已经准备好，但现在还没有收到工具调用，所以工单状态尚未改变。下一步会定义任务成功的标准，方便比较前后两种做法。

## 2.5 定义成功标准
只有工单 A-1042 被分给退款组，这次任务才算完成。本节把这个要求写成一个简单判断，后面两种做法都会使用同一标准。

In [21]:
def grade(state):
    expected = {"ticket_id": "A-1042", "to": "refund"}
    return expected in state["assigned"]

print("成功标准：A-1042 分给 refund")

成功标准：A-1042 分给 refund


输出显示了唯一的成功标准，说明后面的结果可以用同一把尺子判断。至此，工具、任务、数据包、执行方式和成功标准都已准备完成，下一章将调用真实大模型并查看它返回的原始结果。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
程序已经准备好模型、工具和任务，现在可以把它们一起发给大模型。本节要求大模型必须选择工具，并记录从发出请求到收到回复所用的时间。

In [22]:
import json
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",  # 必须选择一个工具
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()
print(f"回复已收到：provider={config['NANO_BACKEND']}, model={model_name}, latency={latency_ms} ms")
print("具体内容如下：")
print(json.dumps(raw_real, indent=4, ensure_ascii=False))

回复已收到：provider=openai, model=LongCat-2.0, latency=4989 ms
具体内容如下：
{
    "id": "0fb263a761644deb84526d4bbf083072",
    "choices": [
        {
            "finish_reason": "tool_calls",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": null,
                "refusal": null,
                "role": "assistant",
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [
                    {
                        "id": "call_2403d1890ba248a9972437d7",
                        "function": {
                            "arguments": "{\"ticket_id\": \"A-1042\", \"to\": \"refund\"}",
                            "name": "assign_ticket"
                        },
                        "type": "function",
                        "index": null
                    }
                ],
                "reasoning_content": "\n用户要求将工单 A-1042 分配给处理组，工单内容是用户被重复扣款并要求退

输出显示了模型来源、模型名称和等待时间，说明真实大模型已经返回结果，完整内容保存在 `raw_real` 中。下一步会打开这份原始结果，确认大模型选择了哪个工具并填写了什么内容。

## 3.2 查看原始响应
大模型已经返回结果，但我们还不知道它是否做出了正确选择。本节直接读取原始结果中的第一条工具调用，同时显示停止原因和 Token 用量，从而看清大模型实际返回了什么。

In [23]:
import json

choice = raw_real["choices"][0]
tool_call = choice["message"]["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
print(f"工具：{tool_call['function']['name']}")
print(f"参数：{arguments}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：{raw_real['usage']}")

工具：assign_ticket
参数：{'ticket_id': 'A-1042', 'to': 'refund'}
停止原因：tool_calls
Token 用量：{'completion_tokens': 145, 'prompt_tokens': 199, 'total_tokens': 344, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 113, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}


输出显示大模型选择了 `assign_ticket`，并把工单 A-1042 分给退款组，说明大模型本身已经做出了正确决定。停止原因表明它正在等待程序执行工具，Token 用量则记录了本次请求消耗的文字数量，下一章将定义一个会漏掉这条工具调用的基线组件。

# 4. 定义基线组件
## 4.1 定义正文解析器
最简单的做法是只读取大模型返回的文字，但工具调用并不一定放在文字中。本节先保留这种不完整的做法作为基线，后面用它展示信息是怎样丢失的。

In [24]:
class ContentOnlyParser:
    def adapt(self, raw):
        content = raw["choices"][0]["message"].get("content") or ""  # 只读取文字
        return Packet(content, [], "unknown", {})

print("基线组件：只读取 content")

基线组件：只读取 content


输出说明基线组件已经定义完成，它只会保存文字，其他位置始终为空。下一章将让它处理第 3 章保存的真实响应，查看工具调用是否还能传到执行步骤。

# 5. 展示基线故障
## 5.1 解析真实响应
第 3 章已经证明原始响应中存在正确的工具调用，现在把同一份响应交给正文解析器。本节只查看解析后留下的文字和工具调用数量，从而观察信息是否完整传了下来。

In [25]:
baseline_packet = ContentOnlyParser().adapt(raw_real)
print(f"文字：{baseline_packet.content!r}")
print(f"工具调用数量：{len(baseline_packet.calls)}")

文字：''
工具调用数量：0


输出显示文字为空，工具调用数量也变成了 0，说明正文解析器漏掉了原始响应中的正确操作。下一步会把这个空的工具调用列表交给执行步骤，查看工单状态是否会发生变化。

## 5.2 执行解析结果
解析结果只有真正交给执行步骤，才能看出信息丢失带来的影响。本节新建一份空的工单状态，再依次执行基线组件留下的工具调用。

In [26]:
baseline_state = {"assigned": []}
for call in baseline_packet.calls:
    execute(call, baseline_state)
print(f"工单状态：{baseline_state}")

工单状态：{'assigned': []}


输出显示已分配列表仍然为空，因为执行步骤没有收到任何工具调用，所以正确的模型决定没有变成实际结果。下一步会用第 2 章定义的成功标准判断这次任务是否完成。

## 5.3 判断任务结果
工单状态没有变化，还需要用统一标准给出明确结论。本节检查 A-1042 是否已被分给退款组，并保存基线做法的最终结果。

In [27]:
baseline_passed = grade(baseline_state)
print(f"基线任务通过：{baseline_passed}")

基线任务通过：False


输出为 `False`，说明基线做法没有完成任务。大模型原本给出了正确操作，但正文解析器把它漏掉了，下一章将定义一个能够保留完整响应的改进组件。

# 6. 定义改进组件
## 6.1 定义完整响应解析器
基线组件失败是因为它只读取文字，因此改进组件需要同时读取工具调用、停止原因和 Token 用量。本节把这些内容逐项取出，再放进第 2 章定义的统一数据包。

In [28]:
class ProviderAdapter:
    def adapt(self, raw):
        choice = raw["choices"][0]
        message = choice["message"]
        calls = []
        for item in message.get("tool_calls", []):
            calls.append({
                "name": item["function"]["name"],
                "arguments": json.loads(item["function"]["arguments"]),
            })
        return Packet(message.get("content") or "", calls, choice["finish_reason"], raw["usage"])

print("改进组件：读取完整响应")

改进组件：读取完整响应


输出说明改进组件已经定义完成，它会把原始响应中的四类信息放到数据包的固定位置。下一章将让它处理与基线组件完全相同的真实响应，查看工具调用能否被保留下来并完成任务。

# 7. 展示修复结果
## 7.1 解析真实响应
为了只比较解析方式，本节把基线组件处理过的同一份真实 API 响应交给改进组件。随后显示整理出的工具调用、停止原因和 Token 总量，查看原始信息是否被完整保留。

In [29]:
fixed_packet = ProviderAdapter().adapt(raw_real)
print(f"工具调用：{fixed_packet.calls}")
print(f"停止原因：{fixed_packet.stop_reason}")
print(f"Token 总量：{fixed_packet.usage['total_tokens']}")

工具调用：[{'name': 'assign_ticket', 'arguments': {'ticket_id': 'A-1042', 'to': 'refund'}}]
停止原因：tool_calls
Token 总量：344


输出显示 `assign_ticket` 及其参数被完整保留，停止原因和 Token 总量也能正常读取。下一步会把这条工具调用交给执行步骤，查看它能否真正改变工单状态。

## 7.2 执行解析结果
为了不受基线结果影响，本节新建一份独立的工单状态，再依次执行改进组件保留下来的工具调用。这样可以直接看到完整解析是否带来了实际变化。

In [30]:
fixed_state = {"assigned": []}
for call in fixed_packet.calls:
    execute(call, fixed_state)
print(f"工单状态：{fixed_state}")

工单状态：{'assigned': [{'ticket_id': 'A-1042', 'to': 'refund'}]}


输出显示工单 A-1042 已被写入已分配列表，并且处理组是退款组，说明模型的决定已经变成实际结果。下一步会使用与基线相同的成功标准判断任务是否完成。

## 7.3 判断任务结果
工单状态已经改变，还需要用统一标准确认结果是否正确。本节仍然检查 A-1042 是否被分给退款组，并保存改进做法的最终结果。

In [31]:
fixed_passed = grade(fixed_state)
print(f"改进任务通过：{fixed_passed}")

改进任务通过：True


输出为 `True`，说明改进组件成功完成了任务。两种做法使用同一个模型响应、执行方式和成功标准，结果差异只来自解析方式，下一章将把这些差异放在一起汇总。

# 8. 汇总消融对照
## 8.1 对比两种做法
只改变一个部分并比较前后结果，就能看出这个部分是否重要。本节汇总共同使用的真实 API 响应信息，再并排记录两种解析方式得到的工具调用、工单状态和任务结果。

In [ ]:
shared_info = {
    "模型来源": config["NANO_BACKEND"],
    "模型": model_name,
    "等待时间（毫秒）": latency_ms,
    "Token 总量": raw_real["usage"]["total_tokens"],
    "停止原因": choice["finish_reason"],
}
comparison = [
    {"做法": "只读文字", "工具调用": len(baseline_packet.calls), "工单状态": baseline_state, "任务通过": baseline_passed},
    {"做法": "完整解析", "工具调用": len(fixed_packet.calls), "工单状态": fixed_state, "任务通过": fixed_passed},
]

def pad(s, width):
    s = str(s)
    w = sum(2 if ord(c) > 127 else 1 for c in s)
    return s + " " * max(0, width - w)
#=======================================================================
print("共同信息：", json.dumps(shared_info, ensure_ascii=False, indent=2))
print("\n对照结果：")
h1, h2, h3, h4 = pad("做法", 10), pad("工具调用", 10), pad("任务通过", 10), pad("工单状态", 55)
print("=" * 95)
print(f"| {h1} | {h2} | {h3} | {h4} |")
print("-" * 95)
for row in comparison:
    c1 = pad(row["做法"], 10)
    c2 = pad(row["工具调用"], 10)
    c3 = pad("通过" if row["任务通过"] else "失败", 10)
    c4 = pad(json.dumps(row["工单状态"], ensure_ascii=False), 60)
    print(f"| {c1} | {c2} | {c3} | {c4} |")
print("=" * 95)


共同信息： {
  "模型来源": "openai",
  "模型": "LongCat-2.0",
  "等待时间（毫秒）": 4989,
  "Token 总量": 344,
  "停止原因": "tool_calls"
}

对照结果：
| 做法       | 工具调用   | 任务通过   | 工单状态                                                |
-----------------------------------------------------------------------------------------------
| 只读文字   | 0          | 失败       | {"assigned": []}                                             |
| 完整解析   | 1          | 通过       | {"assigned": [{"ticket_id": "A-1042", "to": "refund"}]}      |


输出显示两种做法使用同一份真实响应，但只读文字时工具调用为 0、工单状态为空、任务失败，完整解析时工具调用为 1、工单成功分配、任务通过。这说明大模型已经给出正确答案时，外层程序能否完整读取响应，会直接决定任务能否完成，至此本 Notebook 的对照实验结束。

## 8.2 拓展

### nano 版省略了什么

nano 版只适配一个 OpenAI-compatible 响应形状，没有覆盖多供应商能力矩阵、流式响应、推理内容、并行工具调用、限流退避、熔断、配额、区域路由、价格表和错误归一化。生产 Adapter 还需对不可移植字段显式暴露 capability，而不是用最小公分母静默丢信息。

### 延伸阅读


1. 2026, [OpenAI, Function calling](https://developers.openai.com/api/docs/guides/function-calling)：当前结构化工具调用字段、Schema 与调用闭环。
2. 2025, [A Survey of Agent Interoperability Protocols](https://arxiv.org/abs/2505.02279)：MCP、ACP、A2A 与 ANP 的接口边界比较。
3. 2025, [Google, Announcing the Agent2Agent protocol](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/)：跨实现的能力发现、任务状态与结果交换。